In [40]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline

In [41]:
PROJECT_ROOT=Path("../../").resolve()
DATA_PATH = PROJECT_ROOT/"ml"/"outputs"/"model_features.csv"
MODEL_DIR = PROJECT_ROOT/"ml"/"models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "isolation_forest.pkl"

In [42]:
df = pd.read_csv(DATA_PATH)

In [43]:
invoice_ids = df["invoice_id"].copy()
X = df.drop(columns=["invoice_id"])
print("X shape:", X.shape)
print("Number of features:", X.shape[1])

X shape: (85410, 35)
Number of features: 35


In [44]:
X = X.apply(pd.to_numeric, errors="coerce")


In [45]:
missing_values = X.isna().sum()
missing_values[missing_values > 0]

Series([], dtype: int64)

In [46]:
infinite_values = np.isinf(X).sum()
infinite_values[infinite_values > 0]

Series([], dtype: int64)

In [47]:
X = X.replace([np.inf, -np.inf], np.nan)
print("Infinite values handled.")

Infinite values handled.


In [48]:
preprocessor = SimpleImputer(
    strategy="median"
)

In [49]:
isolation_forest = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    max_samples="auto",
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

In [50]:
model_pipeline = Pipeline([
    ("imputer", preprocessor),
    ("isolation_forest", isolation_forest)
])

In [51]:
model_pipeline.fit(X)

,steps,"[('imputer', ...), ('isolation_forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,n_estimators,300


In [52]:
predictions = model_pipeline.predict(X)


In [53]:
raw_scores = model_pipeline.decision_function(X)
print(raw_scores[:10])

[ 0.07864232  0.07618752  0.02453778  0.10484444  0.03114927  0.06870606
  0.0877435   0.05271109  0.01972058 -0.01092768]


In [54]:
anomaly_scores = -raw_scores

In [55]:
results = pd.DataFrame({
    "invoice_id": invoice_ids,
    "anomaly_score": anomaly_scores,
    "is_anomaly": predictions == -1
})
results.head()

,invoice_id,anomaly_score,is_anomaly
0,INV00000001,-0.078642,False
1,INV00000002,-0.076188,False
2,INV00000003,-0.024538,False
3,INV00000004,-0.104844,False
4,INV00000005,-0.031149,False


In [56]:
anomaly_count = results["is_anomaly"].sum()
total_count = len(results)

anomaly_percentage = (
    anomaly_count / total_count
) * 100

print("Total invoices :", total_count)
print("Anomalies      :", anomaly_count)
print("Anomaly %      :", round(anomaly_percentage, 2))

Total invoices : 85410
Anomalies      : 4271
Anomaly %      : 5.0


In [57]:
results["anomaly_score"].describe()

count    85410.000000
mean        -0.072593
std          0.037035
min         -0.143670
25%         -0.099598
50%         -0.080898
75%         -0.052863
max          0.113332
Name: anomaly_score, dtype: float64

In [58]:
high_threshold = results["anomaly_score"].quantile(0.95)
medium_threshold = results["anomaly_score"].quantile(0.80)

def assign_risk(score):
    if score >= high_threshold:
        return "HIGH"
    elif score >= medium_threshold:
        return "MEDIUM"
    else:
        return "LOW"

results["ml_risk"] = results["anomaly_score"].apply(assign_risk)

In [59]:
results["rank_percentile"] = (
    results["anomaly_score"]
    .rank(pct=True) * 100
)

In [60]:
top_anomalies = (
    results
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(20)
)

top_anomalies

,invoice_id,anomaly_score,is_anomaly,ml_risk,rank_percentile
30458,INV00030459,0.113332,True,HIGH,100.000000
627,INV00000628,0.107630,True,HIGH,99.998829
66352,INV00066353,0.106127,True,HIGH,99.997658
4342,INV00004343,0.104655,True,HIGH,99.996488
49517,INV00049518,0.104188,True,HIGH,99.995317
82927,INV00082928,0.099889,True,HIGH,99.994146
26590,INV00026591,0.099443,True,HIGH,99.992975
41601,INV00041602,0.097775,True,HIGH,99.991804
82607,INV00082608,0.097090,True,HIGH,99.990633
27630,INV00027631,0.095644,True,HIGH,99.989463


In [61]:
results["ml_risk"].value_counts()
results["ml_risk"].value_counts(normalize=True) * 100

ml_risk
LOW       80.000000
MEDIUM    14.999415
HIGH       5.000585
Name: proportion, dtype: float64

In [62]:
ANOMALY_OUTPUT = PROJECT_ROOT/"ml"/"outputs"/"anomaly_scores.csv"
results.to_csv(
    ANOMALY_OUTPUT,
    index=False
)


In [63]:
ANOMALY_OUTPUT = PROJECT_ROOT / "ml" / "outputs" / "anomaly_scores.csv"

results.to_csv(
    ANOMALY_OUTPUT,
    index=False
)

print("Saved:", ANOMALY_OUTPUT)

Saved: C:\Users\kshit\OneDrive\Desktop\Hack\ml\outputs\anomaly_scores.csv


In [64]:
joblib.dump(
    model_pipeline,
    MODEL_PATH
)

print("Model saved:", MODEL_PATH)

Model saved: C:\Users\kshit\OneDrive\Desktop\Hack\ml\models\isolation_forest.pkl


In [65]:
saved_model = joblib.load(MODEL_PATH)
print(type(saved_model))
print(saved_model.named_steps)

<class 'sklearn.pipeline.Pipeline'>
{'imputer': SimpleImputer(strategy='median'), 'isolation_forest': IsolationForest(contamination=0.05, n_estimators=300, n_jobs=-1,
                random_state=42)}


In [66]:
test_predictions = saved_model.predict(X.head(10))
print(test_predictions)

[ 1  1  1  1  1  1  1  1  1 -1]


In [67]:
test_scores = -saved_model.decision_function(X.head(10))
print(test_scores)

[-0.07864232 -0.07618752 -0.02453778 -0.10484444 -0.03114927 -0.06870606
 -0.0877435  -0.05271109 -0.01972058  0.01092768]


In [68]:
print("=" * 60)
print("ISOLATION FOREST TRAINING SUMMARY")
print("=" * 60)

print(f"Total invoices       : {len(results):,}")
print(f"Features used        : {X.shape[1]}")
print(f"Anomalies detected   : {anomaly_count:,}")
print(f"Anomaly percentage   : {anomaly_percentage:.2f}%")
print(f"High risk invoices   : {(results['ml_risk'] == 'HIGH').sum():,}")
print(f"Medium risk invoices : {(results['ml_risk'] == 'MEDIUM').sum():,}")
print(f"Low risk invoices    : {(results['ml_risk'] == 'LOW').sum():,}")



ISOLATION FOREST TRAINING SUMMARY
Total invoices       : 85,410
Features used        : 35
Anomalies detected   : 4,271
Anomaly percentage   : 5.00%
High risk invoices   : 4,271
Medium risk invoices : 12,811
Low risk invoices    : 68,328
